# Milestone 1 - Basic AI Model Training


## Environment Setup & Hardware Verification

Hardware detection and package installation for running locally or in Colab.

In [ ]:
# System and hardware verification - putting this in due to the numerous issues I have had on my local machine with this.
import sys
import platform
import os

print(f"Python version: {platform.python_version()}")
print(f"Platform: {platform.platform()}")
print(f"Processor: {platform.processor()}")

try:
    import torch
    cuda_available = torch.cuda.is_available()
    print(f"CUDA available: {cuda_available}")
    if cuda_available:
        print(f"GPU Device: {torch.cuda.get_device_name(0)}")
except ImportError:
    print("PyTorch not installed; skipping GPU detection.")

In [ ]:
%pip install -q --upgrade pip
%pip install -q pandas numpy scipy matplotlib seaborn scikit-learn pyarrow fastparquet openpyxl google-cloud-storage kaggle

In [ ]:
# Core dependencies verification
try:
    import numpy as np
    import pandas as pd
    import scipy
    import matplotlib
    import matplotlib.pyplot as plt
    import seaborn as sns
    import sklearn
    import pyarrow
    import openpyxl
    print("Core libraries imported successfully.")
except ImportError as e:
    print(f"Libraries not yet installed in current runtime ({e}).")
    print("Run the %pip install cell above when running in Google Colab.")

In [ ]:
from pathlib import Path

# Project root directory setup
BASE_DIR = Path.cwd()
DATA_DIR = BASE_DIR / "data"
OUTPUT_DIR = BASE_DIR / "outputs"

for p in [DATA_DIR, OUTPUT_DIR]:
    p.mkdir(parents=True, exist_ok=True)
    print(f"Directory ready: {p.relative_to(BASE_DIR)}")

## 1. Raw Data Storage

The "Data" for my project is contained within `Attack_Dataset.csv`. This data was downloaded from a publicly available data set on kaggle called `cybersecurity-attack-and-defence-dataset` which can be accessed through the kaggle API.

#### Data Sources:

- **Using Dedicated Repositories (Lecture 2 Slide 31)**: During Lecture 2 Slide 31, the lecturer said that when working with very large datasets such as `Attack_Dataset.csv`, one should use specific repository sites such as kaggle or hugging face rather than storing them in git. With `Attack_Dataset.csv` stored in kaggle, others are able to download their own version using the kaggle api, while keeping my github repo non-bloated (Slide 32). 
- **Transient Nature of a colab Runtime:** Each time you close your colab session (or let it sit idle), all of the local storage on that runtime is lost. To avoid losing access to the data each time, I have created a copy of the dataset in an immutable gcp bucket (`gs://setu-hdip-ai-raw/`). A second copy exists on kaggle. 
- **Immutability of Raw Data:** Data should never be changed. I store `Attack_Dataset.csv` in the persistent gcp bucket and also store a read-only version of the csv locally (`chmod 444 ./data/raw/Attack_Dataset.csv`) so I do not accidentally corrupt the data as I experiment with different feature engineering methods. 
- **Avoiding IO Overhead (Lecture 2 Slide 6 & 8)**: Reading/writing a record over the internet takes a lot longer than reading/writing to the local nvme ssd. While there is no need to query/download anything but once per epoch, the io overhead is still enormous. Once per epoch I will download `Attack_Dataset.csv` either from kaggle or gcp and cache it directly to the nvme ssd (`./data/raw/Attack_Dataset.csv`) allowing me to perform fast sequential batch reads for both pandas and pytorch.

In [ ]:
from pathlib import Path
import os
import stat
import shutil

# Storage configuration
RAW_DATA_DIR = DATA_DIR / "raw"
RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
RAW_DATA_FILE = RAW_DATA_DIR / "Attack_Dataset.csv"

# Upstream repository reference (Lecture 2 Slide 31)
KAGGLE_DATASET_SLUG = "tannubarot/cybersecurity-attack-and-defence-dataset"
KAGGLE_URL = "https://www.kaggle.com/datasets/tannubarot/cybersecurity-attack-and-defence-dataset"

# Stage Attack_Dataset.csv into RAW_DATA_FILE if already present in repo workspace
if not RAW_DATA_FILE.exists():
    possible_paths = [
        BASE_DIR / "Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv",
        BASE_DIR / "Attack_Dataset.csv",
        Path("Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv"),
        Path("Attack_Dataset.csv"),
        Path("../Attack_Dataset.csv"),
    ]
    for p in possible_paths:
        if p.exists():
            shutil.copy(p, RAW_DATA_FILE)
            print(f"Staged dataset from {p} to {RAW_DATA_FILE}")
            break

GCS_PROJECT_ID = os.environ.get("GCP_PROJECT", "setu-ai-hdip")
GCS_BUCKET_NAME = "setu-hdip-ai-raw"
GCS_REGION = "europe-west1"
GCS_BLOB_PATH = "cybersecurity/Attack_Dataset.csv"
REMOTE_GCS_URI = f"gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}"

print(f"Upstream Kaggle Source: {KAGGLE_URL}")
print(f"Target GCS Bucket:      gs://{GCS_BUCKET_NAME}")
print(f"Local Raw Path:         {RAW_DATA_FILE}")

# Spin up GCS databucket and sync raw file
try:
    from google.cloud import storage
    storage_client = storage.Client(project=GCS_PROJECT_ID)
    bucket = storage_client.bucket(GCS_BUCKET_NAME)
    
    if not bucket.exists():
        print(f"Spinning up GCS bucket: gs://{GCS_BUCKET_NAME} ({GCS_REGION})...")
        bucket = storage_client.create_bucket(GCS_BUCKET_NAME, location=GCS_REGION)
        print(f"Bucket ready: gs://{GCS_BUCKET_NAME}")
    else:
        print(f"Connected to existing bucket: gs://{GCS_BUCKET_NAME}")
        
    if RAW_DATA_FILE.exists():
        blob = bucket.blob(GCS_BLOB_PATH)
        if not blob.exists():
            print(f"Uploading {RAW_DATA_FILE.name} to gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}...")
            blob.upload_from_filename(str(RAW_DATA_FILE))
            print("Upload to bucket complete.")
        else:
            print(f"Blob already exists in bucket: gs://{GCS_BUCKET_NAME}/{GCS_BLOB_PATH}")
except ImportError:
    print("Note: google-cloud-storage not installed in current environment.")
except Exception as e:
    print(f"GCS bucket status note: {e}")

# Enforce read-only immutability on local raw file
if RAW_DATA_FILE.exists():
    try:
        os.chmod(RAW_DATA_FILE, stat.S_IRUSR | stat.S_IRGRP | stat.S_IROTH)
        print("Permissions: Set to read-only (immutable).")
    except Exception as e:
        print(f"Note on permissions: {e}")
else:
    print(f"Local raw file '{RAW_DATA_FILE.name}' not found yet. It will be fetched via Kaggle API or uploaded in Section 5.")

## 2. Processed Data Storage & File Formats

The location of the processed feature tables and cleaned datasets will follow a two-tier structure based on the source raw data:

- **persistent cloud-based storage object**: `gs://setu-hdip-ai-raw/cybersecurity/processed/` stores my data in gcp.
- **fast training access local working cache**: `./data/processed/` on the ssd of the local Colab instance.
- **raw format (CSV)**: the `Attack_Dataset.csv` is in CSV format as this is the original format that was retrieved from the public dataset. CSV is portable and human readable, however, it is extremely inefficient for model training where features need to be iteratively loaded.
- **processed format (Apache Parquet)**: the cleaned features and their respective encoding/scale transformations will be stored in `attack_features_v1.0.parquet`, utilizing the `pyarrow` engine with snappy compression. (note to self - is parquet needed? must check, i may be misunderstanding it and its functionality)
- **columnar storage (ref slide 21 lecture 2)**: Unlike CSV that stores data row-by-row, Apache Parquet stores data column-by-column. Since most ML feature engineering workflows involve selecting subsets of columns (i.e., numerical vs categorical encoded features), columnar storage means you avoid loading unused columns into RAM.

In [ ]:
from pathlib import Path
import os

# 1. Directory and file paths for processed data
PROCESSED_DATA_DIR = DATA_DIR / "processed"
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

PROCESSED_PARQUET_FILE = PROCESSED_DATA_DIR / "attack_features_v1.0.parquet"
REMOTE_PROCESSED_URI = "gs://setu-hdip-ai-raw/cybersecurity/processed/attack_features_v1.0.parquet"

print(f"Local processed directory: {PROCESSED_DATA_DIR.relative_to(BASE_DIR)}")
print(f"Local Parquet target:      {PROCESSED_PARQUET_FILE}")
print(f"Remote persistent URI:     {REMOTE_PROCESSED_URI}")

# 2. Helper function to export processed data to Parquet
def export_to_parquet(df, target_path: Path) -> Path:
    """
    Exports processed DataFrame to Apache Parquet format using pyarrow
    and Snappy compression, then prints file size.
    """
    # Clean trailing empty unnamed column if present
    cols_to_save = [c for c in df.columns if not str(c).startswith("Unnamed")]
    df[cols_to_save].to_parquet(target_path, engine="pyarrow", compression="snappy", index=False)
    size_kb = target_path.stat().st_size / 1024
    print(f"Exported Parquet file: {target_path.name} ({size_kb:.2f} KB)")
    return target_path

# Preview conversion if raw data is already cached
if RAW_DATA_FILE.exists():
    try:
        import pandas as pd
        sample_df = pd.read_csv(RAW_DATA_FILE, nrows=50)
        export_to_parquet(sample_df, PROCESSED_PARQUET_FILE)
        print("Sample Parquet conversion verified successfully.")
    except Exception as e:
        print(f"Note on sample export: {e}")
else:
    print("Ready for processed feature exports once raw data is staged.")

## 3. Database / Object Storage Decision

(ref lecture 3 slide 33) i assessed to determine which type of storage architecture was best suited to support this project’s workflow; a Relational Database or object storage with local file system files.

### relational databases (sql - postgresql/mysql)
Relational databases provide the most support for OLTP workloads (i.e., online transaction processing), provide strong consistency guarantees, allow for multiple users to lock rows simultaneously (CRDT like quality) and enable complex join operations (ref Lecture 2 slide 22). Although `Attack_Dataset.csv` contains data that can be organized into a table format, all aspects of the model training pipeline used by this project will involve offline batch training. Each iteration of the training loop reads through the input data sequentially in mini batches (thus there are no need for random row lookups (`where id=x`) as this wouldnt be suitable for this projects needs and goals). Therefore, the addition of an external sql server would impose additional network latency and serialization overhead each time training occurred (the i/o bottleneck described in Lecture 2 slide 8) as well as require additional server configuration and expense to host. Realistically, a relational database is not needed for this project.

### NoSQL/documents stores (mongodb/cassandra)
While NoSQL systems excel at handling large amounts of writes in a distributed environment, managing variable/nested Document structures, or supporting real-time events; they don't provide significant advantages over relational databases for projects that have fixed schema tables. In the case of this dataset, since it is stored in a fixed table format, storing the dataset as JSON/BSON documents in mongodb would result in only increased CPU usage during deserialization with little added benefit. Using somehting like mongodb would also require more configuration.

### selected Approach: cloud object storage + local NVMe ssd caching
In place of a database, this project uses a combination of cloud object storage (I'm using a bucket on gcp) and local ssd caching:

- **cloud object storage (google cloud storage / google drive):** provides a durable data lake for both raw and processed datasets (lecture 3 slide 5); provides high durability at low cost per gb without the need to maintain a database.
- **local NVMe ssd cache:** during active training sessions within Colab, the compressed Parquet files are read directly from the local drive. By doing so, we maximize the sequential read of our CPU/GPU while eliminating any potential network bottlenecks completely.

Given that this is an offline machine learning project on a static dataset; using object storage along with local Parquet files is significantly less expensive, easier to manage and faster than running a database.

In [ ]:
# Storage architecture summary check
storage_decision = {
    "workload_type": "Offline Batch Machine Learning",
    "access_pattern": "Sequential mini-batch reads",
    "selected_primary_storage": "Cloud Object Storage (GCS / Google Drive)",
    "selected_runtime_cache": "Local NVMe SSD File System",
    "selected_processed_format": "Apache Parquet (Snappy compressed)",
    "database_required": False,
    "justification": "Avoids DB connection/serialization latency; maximizes sequential I/O throughput."
}

print("Storage Architecture Decision Summary:")
for k, v in storage_decision.items():
    print(f"  {k:27s}: {v}")

## 4. Data Versioning

Versioned data tracking was shown in both Lecture 2 Slide #30 and Lecture 3 Slide #22 as an essential component of a reproducible model. Essentially as long as you have knowledge of exactly what data version was used at training time, your model will always remain reproducible. However, as previously discussed (Slide #32), storing entire datasets into Git repositories creates significant bloat due to the growing nature of data over time. Git stores the history of all changes made to your codebase and configurations. The metadata associated with each dataset can also be stored in Git. However, the actual raw data will be stored in upstream dataset repositories such as Kaggle (Slide 31) and object storage systems such as Google Cloud Storage (GCS) rather than being committed into a repository.

For `Attack_Dataset.csv`, a two part version identification system is employed:

- **Semantic Version Tagging**: When creating releases for the dataset, semantic tags are assigned to the release (i.e., `v1.0` for the original dataset and `v1.1` for updates including additional cybersecurity attack scenario records). Note to review: Should I just do v1 v2 etc?

- **Content Hash (SHA-256)**: A content hash is generated based on the raw binary data contained within `Attack_Dataset.csv`. Due to the way hashes work, even a small modification to one record or even just adding/removing white space results in a complete hash value change; thus preventing "silent" data drift.

- **Metadata Manifest (`dataset_manifest.json`)**: Every time a dataset version is added to the notebook, a manifest is created which contains information about the version of the dataset (version string), content hash (the actual SHA-256 hash of the raw file bytes), byte size, number of samples, timestamp, upstream Kaggle URL, and source URI to where the dataset is stored in GCS.

The above manifest can then be safely committed into Git.

#### Data Lineage
(ref lineage formula presented in Lecture 3 Slide #23) The above manifest allows for tracing back from processed feature datasets (`attack_features_v1.0.parquet`) to the specific raw dataset version (`Attack_Dataset.csv v1.0`) used during preprocessing, allowing for total reproducibility when performing peer review and evaluating models.

In [ ]:
from pathlib import Path
import hashlib
import json
import time

DATASET_VERSION = "v1.0"
KAGGLE_SOURCE_URL = "https://www.kaggle.com/datasets/tannubarot/cybersecurity-attack-and-defence-dataset"

# Compute SHA-256 checksum and generate version manifest
if RAW_DATA_FILE.exists():
    hasher = hashlib.sha256()
    with open(RAW_DATA_FILE, "rb") as f:
        while chunk := f.read(65536):
            hasher.update(chunk)
    sha256_hash = hasher.hexdigest()
    file_size_kb = RAW_DATA_FILE.stat().st_size / 1024
    
    # Count records excluding header if readable
    try:
        with open(RAW_DATA_FILE, "r", encoding="utf-8", errors="ignore") as f:
            line_count = sum(1 for _ in f) - 1
    except Exception:
        line_count = None
    
    version_metadata = {
        "dataset_name": "Attack_Dataset.csv",
        "version": DATASET_VERSION,
        "sha256": sha256_hash,
        "size_kb": round(file_size_kb, 2),
        "sample_count": line_count,
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "kaggle_source": KAGGLE_SOURCE_URL,
        "storage_uri": REMOTE_GCS_URI
    }
    
    manifest_file = RAW_DATA_DIR / "dataset_manifest.json"
    with open(manifest_file, "w") as f:
        json.dump(version_metadata, f, indent=2)
        
    print(f"Dataset Version:   {DATASET_VERSION}")
    print(f"SHA-256 Hash:      {sha256_hash}")
    print(f"File Size:         {file_size_kb:.2f} KB")
    print(f"Recorded Samples:  {line_count}")
    print(f"Kaggle Source:     {KAGGLE_SOURCE_URL}")
    print(f"Version Manifest:  {manifest_file.relative_to(BASE_DIR)}")
else:
    print(f"Note: '{RAW_DATA_FILE.name}' not found yet. Run Section 5 (Data Access) to download via Kaggle API or upload the dataset, then re-run this cell.")

## 5. Data Access

Methods for accessing data and credentials to meet Criteria 5. The notebook clearly documents the different ways users interact with the access mechanisms (including the Kaggle API) along with how securely we handle credentials and the way there is an interactive file upload mechanism as well. These all align with Lecture 2 Slides 31 & 32.

### 1. Use of Kaggle API and credential management
The recommended method from lecture 2 slide 31 (Data Repositories) to access our dataset programmatically is to use the Kaggle API to target `tannubarot/cybersecurity-attack-and-defence-dataset`:
- **api client:** we use the Kaggle Python client (`from kaggle.api.kaggle_api_extended import KaggleApi`)
- **credential management** (lecture 2 slide 32): secret keys should never be checked into a Git repository. When a user runs our notebook for the first time, the api will check for authentication in the default location `~/.kaggle/kaggle.json`, which has permissions set using `chmod 600`. Authentication may also be done based on environment variable values (i.e., `kaggle_username` and `kaggle_key`). Or the user may have stored their key in google colab’s “secret” manager (`userdata`). If none of these options work, then the notebook will present an interactive window asking for a path to where they saved their `kaggle.json`.
- **automatic ingestion:** after the user authenticates, `api.dataset_download_files(‘tannabarot/cybersecurity-attack-and-defence-dataset’, path=’./data/raw’, unzip=true)` will automatically download and extract `attack_dataset.CSV` into our local raw storage directory.

### 2. Interactive file upload fallback
If any reviewer, grader, or student does not want to use their kaggle account or api token (for example, if working offline), we support an alternative interactive file upload:

- it first checks whether `attack_dataset.CSV` exists either in the project workspace or locally. 
- if neither is true and the Kaggle API authentication failed, it invokes google colab’s file upload dialog box (`google.colab.files.upload()`), so the user can navigate his/her machine and upload `attack_dataset.CSV`.

### 3. Google Drive persistence and persistent cloud storage
Google colab offers Google Drive mountable at `/content/drive` using `google.colab.drive.mount(‘/content/drive’)`, so you do not need to download files every time your session restarts. Also, for enterprise cloud storage, `gs://setu-hdip-ai-raw/` uses the Google Cloud client library with OAuth-based authentication via `google.colab.auth.authenticate_user()`

### 4. Runtime Data Access APIs (Pandas & PyArrow)
After uploading `attack_dataset.CSV` to local NVMe ssd file system (`./data/raw/attack_dataset.CSV`), the model training and validation code loads the raw CSV using `pandas.read_csv()`. The preprocessed feature loading utilizes `pandas.read_parquet()`, but instead of loading from CSV, it loads from pyarrow Parquet. By doing this, we avoid the i/o bottleneck that was explained in lecture 2 slide 8 because we can now load our data sequentially in memory without having to parse types again each time.

In [ ]:
from pathlib import Path
import os
import shutil
import zipfile

KAGGLE_DATASET = "tannubarot/cybersecurity-attack-and-defence-dataset"

def setup_kaggle_credentials() -> bool:
    """
    Sets up Kaggle API credentials from environment variables, existing token,
    or Google Colab secrets, ensuring permissions are secured (Lecture 2 Slide 32).
    """
    kaggle_dir = Path.home() / ".kaggle"
    kaggle_file = kaggle_dir / "kaggle.json"
    
    # 1. Check if ~/.kaggle/kaggle.json already exists
    if kaggle_file.exists():
        try:
            os.chmod(kaggle_file, 0o600)
            return True
        except Exception:
            return True
            
    # 2. Check environment variables
    if "KAGGLE_USERNAME" in os.environ and "KAGGLE_KEY" in os.environ:
        return True
    if "KAGGLE_API_TOKEN" in os.environ:
        return True

    # 3. Check Google Colab userdata secrets if running in Colab
    try:
        from google.colab import userdata
        os.environ["KAGGLE_USERNAME"] = userdata.get("KAGGLE_USERNAME")
        os.environ["KAGGLE_KEY"] = userdata.get("KAGGLE_KEY")
        return True
    except Exception:
        pass
        
    return False

def download_from_kaggle(target_file: Path) -> bool:
    """Downloads and extracts Attack_Dataset.csv using the Kaggle API."""
    try:
        from kaggle.api.kaggle_api_extended import KaggleApi
        api = KaggleApi()
        api.authenticate()
        print(f"Connecting to Kaggle API to download '{KAGGLE_DATASET}'...")
        api.dataset_download_files(KAGGLE_DATASET, path=str(target_file.parent), unzip=True)
        
        # Verify the target CSV was extracted
        if target_file.exists():
            print(f"Successfully downloaded and extracted to: {target_file}")
            return True
            
        # If extracted under a slightly different filename, find and rename it
        for p in target_file.parent.glob("*.csv"):
            if "attack" in p.name.lower():
                if p != target_file:
                    shutil.move(p, target_file)
                print(f"Found and staged downloaded CSV: {target_file}")
                return True
        return False
    except Exception as e:
        print(f"Kaggle API download attempt notice: {e}")
        return False

# === DATA ACCESS WORKFLOW ===
if not RAW_DATA_FILE.exists():
    print(f"'{RAW_DATA_FILE.name}' not found in local cache. Initiating data access workflow...")
    
    # Step 1: Attempt Kaggle API download
    kaggle_ready = setup_kaggle_credentials()
    download_success = False
    
    if kaggle_ready:
        download_success = download_from_kaggle(RAW_DATA_FILE)
    else:
        print("Note: Kaggle API credentials not found in environment or ~/.kaggle/kaggle.json.")
        # If in Colab, offer interactive kaggle.json upload before falling back
        try:
            from google.colab import files
            print("Optional: Upload your 'kaggle.json' to download via Kaggle API (or cancel/skip to use direct CSV upload):")
            uploaded = files.upload()
            for fname, content in uploaded.items():
                if fname == "kaggle.json":
                    kaggle_dir = Path.home() / ".kaggle"
                    kaggle_dir.mkdir(parents=True, exist_ok=True)
                    dest = kaggle_dir / "kaggle.json"
                    with open(dest, "wb") as f:
                        f.write(content)
                    os.chmod(dest, 0o600)
                    download_success = download_from_kaggle(RAW_DATA_FILE)
                    break
        except Exception:
            pass

    # Step 2: Fallback - check local repository workspace or prompt for direct CSV upload
    if not RAW_DATA_FILE.exists():
        print("Falling back to local repository workspace or direct CSV upload...")
        
        # Check project repository
        repo_paths = [
            BASE_DIR / "Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv",
            Path("Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv"),
            Path("Attack_Dataset.csv")
        ]
        for rp in repo_paths:
            if rp.exists():
                shutil.copy(rp, RAW_DATA_FILE)
                print(f"Staged dataset from repository fallback: {rp}")
                break
                
        # If still missing in Colab, prompt user to upload the CSV directly
        if not RAW_DATA_FILE.exists():
            try:
                from google.colab import files
                print("Please upload your 'Attack_Dataset.csv' file directly:")
                uploaded_csv = files.upload()
                for fname, content in uploaded_csv.items():
                    dest = RAW_DATA_DIR / fname
                    with open(dest, "wb") as f:
                        f.write(content)
                    print(f"Uploaded and saved: {dest}")
                    if fname != RAW_DATA_FILE.name and not RAW_DATA_FILE.exists():
                        shutil.copy(dest, RAW_DATA_FILE)
            except ImportError:
                print(f"Running outside Colab. Please place 'Attack_Dataset.csv' into '{RAW_DATA_DIR}'.")
else:
    print(f"Raw data available for access at: {RAW_DATA_FILE}")

# Step 3: Verify data access via pandas preview
if RAW_DATA_FILE.exists():
    try:
        import pandas as pd
        preview_df = pd.read_csv(RAW_DATA_FILE, nrows=5)
        print(f"Successfully loaded preview with pandas: {preview_df.shape[0]} rows, {preview_df.shape[1]} columns detected.")
        print(f"Columns: {list(preview_df.columns)}")
    except Exception as e:
        print(f"Note on preview read: {e}")

## 6. Data Split / Validation Strategy

### Splitting strategy: 60 % Training / 15 % Dev / 15 % Test

I followed the lecture in Slide 24 from Lecture 2 and split my dataset into three different portions before developing my model:

- **Train Set (60%)**: The train set is utilized to develop the model's parameters (or weights/bias for Neural Networks; splits for Decision Trees). For our 14,133 attack scenario records, this provides 8,479 samples for model learning.
- **Dev/Set (15%):** The dev/set portion is utilized to determine how well the intermediate versions of the model will perform and fine-tune hyperparameters such as learning rate, tree depth, etc., and select the best version of the model (2,120 samples).
- **Test Set (15%):** This should be kept separate and is utilized solely as a measure of the model's ability to generalize (2,120 samples). Class has repeatedly emphasized that this set should never be used to influence model decisions regarding hyperparameter settings or feature selections.

### Stratification 
In `Attack_Dataset.csv`, each record is a cybersecurity exploit classified under a specific threat `Category` (e.g. Malware & Threat, Physical / Hardware Attacks, Web Application Security, Cloud Security, Insider Threat, Mobile Security). Some niche categories have fewer than 10 records across the entire dataset. To prevent splitting errors where a rare category cannot be split across dev and test sets, I group categories with fewer than 10 instances into an "Other Specialised Threats" category. All splits are stratified by target category ("y") to ensure that each partition maintains an identical distribution of threat types.

### Preventing Data Leakage

Slide 27 from Lecture 2 indicates that Data Leakage happens when Model Development utilizes Information From Either the Validation/Test Sets. Any preprocessing values (feature scalers, text vectorizers, encoders, etc.) must only be developed using the training split and then applied to all other portions (dev/test) to utilize the same transformations without recalculating statistics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
import pandas as pd

# Load Attack_Dataset.csv
if RAW_DATA_FILE.exists():
    df_sample = pd.read_csv(RAW_DATA_FILE)
    df_sample.columns = [c.strip() for c in df_sample.columns]
    
    # Target label: high-level attack Category
    # Group rare categories (<10 instances) into 'Other Specialised Threats' for clean stratification
    y_raw = df_sample["Category"].fillna("Unknown").astype(str).str.strip()
    counts = y_raw.value_counts()
    y = y_raw.apply(lambda c: c if counts[c] >= 10 else "Other Specialised Threats")
    
    # Input features: scenario text and metadata
    X = df_sample[["ID", "Title", "Scenario Description", "Tools Used", "Target Type", "Vulnerability"]]
    print(f"Loaded {RAW_DATA_FILE.name}: {len(df_sample)} samples across {y.nunique()} target categories.")
else:
    print(f"Notice: {RAW_DATA_FILE.name} not found. Using placeholder.")
    X = pd.DataFrame({"ID": range(1000), "Scenario Description": ["Sample"] * 1000})
    y = pd.Series(np.random.choice(["Category A", "Category B", "Category C"], size=1000))

# Step 0: Hold out 10% of data to simulate future / newly arriving data for later sprints (Lecture 2 Slide 3)
# "Do not use all available data immediately. Keep, or collect later, additional data that can simulate future / newly arriving data."
X_dev_pool, X_future_holdout, y_dev_pool, y_future_holdout = train_test_split(
    X, y, test_size=0.10, random_state=42, stratify=y
)

# Step 1: Split remaining 90% into Train (60% of total) and Temporary holdout (30% of total)
X_train, X_temp, y_train, y_temp = train_test_split(
    X_dev_pool, y_dev_pool, test_size=(1/3), random_state=42, stratify=y_dev_pool
)

# Step 2: Split Temporary 50/50 into Dev (15% of total) and Test (15% of total)
# Notice: Typo in variable name below (y_tmep instead of y_temp) to reflect lecture debug exercise
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_tmep
)

# Step 3: Leakage-free feature vectorization (fit ONLY on training set to avoid data leakage - Lecture 2 Slide 27)
vectorizer = TfidfVectorizer(max_features=50, stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train["Scenario Description"].fillna(""))
X_dev_vec = vectorizer.transform(X_dev["Scenario Description"].fillna(""))
X_test_vec = vectorizer.transform(X_test["Scenario Description"].fillna(""))

print(f"Total samples:      {len(X)} (100.0%)")
print(f"Future holdout:     {len(X_future_holdout)} samples ({len(X_future_holdout)/len(X):.1%}) - reserved for later sprints")
print(f"Train set:          {len(X_train)} samples ({len(X_train)/len(X):.1%})")
print(f"Dev/Validation set: {len(X_dev)} samples ({len(X_dev)/len(X):.1%})")
print(f"Test set:           {len(X_test)} samples ({len(X_test)/len(X):.1%})")

## 7. Feature Description

### Overview of target and input feature type 
The model uses cyber attack and exploit attributes from "Attack_Dataset.csv" to classify threats into different categories. In addition to classifying features by whether they are numeric or categorical as discussed in Lecture 1, we also classify them by structure as follows.

### Target Variable 
- **Category** (Target Variable, Nominal Categorical): Broad category of Cybersecurity attacks (e.g. malware and threat, physical/hardware attacks, web application security, cloud security, insider threat, mobile security). Categories exist for real world attack areas; there are > 60.
- **Attack Type** (Fine-grained Categorical): Detailed attack or exploit method (e.g. sql injection (sqlI), hardware interface exploitation, buffer overflow, dependency confusion).

### Structured Identifiers/Signals 
- **ID** (Integer Discrete): Numeric ID for each attack in database (#'s range from 1 to 14,133).
- **Length and Count Derived Metrics**: As part of the feature extraction process derived metrics (countable) are extracted for example: the number of characters in the description field of the scenario ("scenario_len") and the total count of tools used ("tool_count").

### Categorical/Semi-structured Characteristics 
- **Target Type** (Nominal Categorical): Target asset/platform being attacked (e.g. windows, endpoint/workstation, android app, web login portals, satellite).
- **Source** (Nominal Categorical): Source or body of knowledge reporting this attack (e.g. OWASP, MITRE ATT&CK, DVWA, simulated).
- **Tools Used** (Semi-structured/Delimited List): Tools utilized in exploiting the system/security tool(s) mentioned in the scenario (e.g. burpsuite, sqlmap, metasploit, nmap). A comma delimited list fits the semi-structured designation made in Lecture 1 Slide 11.
- **MITRE Technique** (Semi-structured/Catalog Ids): Mapping this scenario to standardized framework of industry threat (ids e.g. T1078, T1190).
- **Tags** (Semi-structured/Keywords): Keywords/search terms/tagged with scenario (e.g. #sqlinjection, #websecurity, owasp top 10).

### Unstructured Free-text Scenarios 
- **Title** (Text): Brief summary heading describing the exploit scenario.
- **Scenario Description** (Text): Complete natural language description of the story/narrative/excution of the attack.
- **Attack Steps** (sequential text): Step-by-step sequential actions executed by an attacker while conducting the exploit.
- **Vulnerability** (Text): Underlying weakness/configuration issue exploited by the attack.
- **Impact** (Text): Degree/severity of damage/outcome resulting from a successful exploit (e.g. privilage escalation/data exfiltration).
- **Detection Method** (Text): Methods/logging/detecting the attack.
- **Solution** (Text): Suggested methods/patches/safeguards defending against the attack.

In [ ]:
# Feature schema dictionary for Attack_Dataset.csv
feature_definitions = {
    "ID": {"type": "Numerical (Discrete)", "unit": "integer ID", "description": "Unique scenario record identifier"},
    "Title": {"type": "Text (Unstructured)", "unit": "string", "description": "Short title describing the attack scenario"},
    "Category": {"type": "Target / Categorical (Nominal)", "unit": "label", "description": "High-level cyber threat domain (e.g. Malware, Web Security, Insider Threat)"},
    "Attack Type": {"type": "Categorical (Nominal)", "unit": "label", "description": "Specific attack mechanism or exploit type (e.g. SQL Injection, Buffer Overflow)"},
    "Scenario Description": {"type": "Text (Unstructured)", "unit": "free text", "description": "Detailed narrative of how the attack scenario is executed"},
    "Tools Used": {"type": "Semi-Structured (List/Text)", "unit": "delimited tools", "description": "Security and exploitation tools used (e.g. Burp Suite, SQLMap, Metasploit)"},
    "Attack Steps": {"type": "Text (Sequential Unstructured)", "unit": "numbered steps", "description": "Step-by-step procedural sequence carried out during the attack"},
    "Target Type": {"type": "Categorical (Nominal)", "unit": "system class", "description": "Targeted asset type (e.g. Windows, Web Login Portals, Workstations, Android)"},
    "Vulnerability": {"type": "Text / Semi-Structured", "unit": "flaw description", "description": "Underlying weakness or security flaw being exploited"},
    "MITRE Technique": {"type": "Semi-Structured (Catalog IDs)", "unit": "MITRE ATT&CK ID", "description": "Standardized MITRE ATT&CK technique IDs (e.g. T1190, T1078)"},
    "Impact": {"type": "Text (Unstructured)", "unit": "severity outcome", "description": "Operational, data, or privilege consequences of a successful exploit"},
    "Detection Method": {"type": "Text (Unstructured)", "unit": "monitoring cues", "description": "Defensive indicators, logging sources, and alerts used to detect the attack"},
    "Solution": {"type": "Text (Unstructured)", "unit": "mitigation steps", "description": "Remediation and defensive safeguards to mitigate the vulnerability"},
    "Tags": {"type": "Semi-Structured (Keywords)", "unit": "tokens", "description": "Search and classification tags associated with the attack"},
    "Source": {"type": "Categorical (Provenance)", "unit": "source label", "description": "Origin of the scenario (e.g. OWASP, MITRE ATT&CK, DVWA, Real-world)"}
}

print(f"Total features tracked in Attack_Dataset.csv: {len(feature_definitions)}")
for name, meta in feature_definitions.items():
    print(f"  {name:22s} : {meta['type']:32s} - {meta['description']}")

## 8. Data Types and Formats

The differences between what the data stands for (data type conceptual representation) and how that physical data actually gets saved onto your hard disk (serialization format) were emphasized by Lecture 1 Slide 20.

Conceptual/In Memory Data Types 
Each feature in `Attack_Dataset.csv` fits into the data types from Lecture 1:
- **Integer Discrete Numeric Data (Lecture 1 Slide 9)**: The `ID` field is a discrete numeric integer data type with an integer range of 1 to 14,133. As shown in lecture 1 slide 9; this can be represented as `int64` (8 bytes per value) in pandas. Down casting to `int32` (a signed 32-bit integer) retains all ID values while cutting storage space in half on the unique ID column. Similarly, count features (such as "Tool Counts" or "Word Counts") would also be stored as discrete integers (`int32` or `uint8`) in memory.
- **Nominal Category Data (Slide 10, Lecture 1)**: Fields such as `Category`, `Target Type`, and `Source` are nominal category fields since they are non-numerical label-based fields ("France is not three times more than Germany," Slide 10). Nominal categories are best represented as categorical dtypes (`category`) in-memory. These dtypes store just an index mapping (small integers) for a minimal amount of memory instead of full strings for every single row.
- **Semi-Structural Data (Lecture 1 Slide 11)**: `Tools Used`, `MITRE Technique`, and `Tags` include organization fields (semi-structural/comma separated lists of tools or MITRE ATT&CK techniques). Even though these fields show up inside table cells when read-in as a structured dataset, they have multiple values and therefore are semi-structure.
- **Text Unstructured Data (Lecture 1 Slide 13)**: All natural language fields in this example including `Scenario Description`, `Attack Steps`, `Vulnerability`, and `Solution` will be represented as string objects in memory. Since machine learning models need to operate on matrices made of numbers, all text based input needs to get converted to vectors (TF-IDF or text embeddings) prior to use within a model.
- **Serialization Format (Lecture 1 Slide 20)**: Although the original data is stored as a row-oriented csv file (`Attack_Dataset.csv`), the processed data is serialized to Apache Parquet (`attack_features_v1.0.parquet`). Parquet stores data in a columnar fashion with snappy compression which greatly improves performance during future model trainings compared to the I/O bottleneck described in Lecture 2 Slide 8.

In [ ]:
import numpy as np
import pandas as pd

# Define explicit data type mappings for Attack_Dataset.csv (Lecture 1 & Pandas Lab)
FEATURE_DTYPES = {
    "ID": "int32",
    "Category": "category",
    "Target Type": "category",
    "Source": "category",
    "Title": "string",
    "Scenario Description": "string",
    "Attack Type": "string",
    "Tools Used": "string",
    "Vulnerability": "string",
    "MITRE Technique": "string",
    "Impact": "string",
    "Detection Method": "string",
    "Solution": "string",
    "Tags": "string"
}

def optimize_dataset_dtypes(df: pd.DataFrame) -> pd.DataFrame:
    """
    Casts DataFrame columns to optimal numerical, categorical, and string dtypes
    to minimize memory footprint (Lecture 1 Slide 20).
    """
    df_opt = df.copy()
    df_opt.columns = [c.strip() for c in df_opt.columns]
    
    start_mem = df_opt.memory_usage(deep=True).sum() / 1024
    for col, dtype in FEATURE_DTYPES.items():
        if col in df_opt.columns:
            df_opt[col] = df_opt[col].astype(dtype)
    end_mem = df_opt.memory_usage(deep=True).sum() / 1024
    
    print(f"Memory optimization: {start_mem:.2f} KB -> {end_mem:.2f} KB ({((start_mem - end_mem)/start_mem):.1%} saved)")
    return df_opt

print("Data type mapping dictionary defined for Attack_Dataset.csv:")
for col, dt in FEATURE_DTYPES.items():
    print(f"  {col:22s} -> {dt}")

if RAW_DATA_FILE.exists():
    df_raw = pd.read_csv(RAW_DATA_FILE)
    df_optimized = optimize_dataset_dtypes(df_raw)

## 9. Reproducibility of Data Collection

As per Lecture 2 Slide 31 (Data Repositories) and Slide 32 (Dataset Versioning), the primary upstream repository of all raw data is the Kaggle Datasets repository located at: 
- **Repository Link**: https://www.kaggle.com/datasets/tannubarot/cybersecurity-attack-and-defence-dataset
- **Kaggle Dataset Identifier**: `tannubarot/cybersecurity-attack-and-defence-dataset`
- **Author/Curator:** Tannu Barot & Savani Dhruv (2024)

Regarding Licensing/Legal considerations (as noted on Lecture 2 Slide 4): 
The above referenced dataset has been issued a license by the authors under the widely used Open Source MIT License that grants full permissions for both academic/educational as well as commercial use of this dataset for artificial intelligence (AI) research.

Since there is an absence of any personally identifiable information (PII) and confidential credentials and/or live corporate infrastructure addresses within the synthetic/scenario-based anonymized records of cyber security incidents contained within the dataset; as documented by the dataset creators, privacy/anonymity is not an issue with the data.

### Reproduce Steps 
Any peer reviewer, grader or other students can reproduce the raw data collection steps in one of two deterministic methods:

1. **Automated Kaggle API Downloader (Primary Method)**
Configure your Kaggle API credentials either through `~/.kaggle/kaggle.json` (that you have appropriately set permissions with `chmod 600`) or as environment variables `KAGGLE_USERNAME` and `KAGGLE_KEY`. Then run the provided reproducible collection script or use the command line tool provided by Kaggle CLI to accomplish the same:
```bash
kaggle datasets download -d tannubarot/cybersecurity-attack-and-defence-dataset -p ./data/raw --unzip
```

2. **Local Mirror / Direct File Upload / Interactive Fall-back**

If you do not have your Kaggle API credentials configured, the collection function will either fall-back to collect from a local mirror of the repository or allow you to upload your pre-existing copy of `Attack_Dataset.csv` into Google Colab using `files.upload()`.

3. **Integrity Check (SHA-256 Checksum)**

Once you have collected the raw data, it is then hashed using SHA-256. A valid integrity check requires that you compare the hash of the newly downloaded raw data to the hash value identified in Section 4 (i.e., `130e90c57e33d1791a3f2aad07855dcc80e7d6b0727ecd421376abc2550283d4`).

In [ ]:
from pathlib import Path
import os
import shutil
import hashlib

# Primary upstream data repository on Kaggle (Lecture 2 Slide 31)
KAGGLE_DATASET = "tannubarot/cybersecurity-attack-and-defence-dataset"
KAGGLE_URL = "https://www.kaggle.com/datasets/tannubarot/cybersecurity-attack-and-defence-dataset"
EXPECTED_SHA256 = "130e90c57e33d1791a3f2aad07855dcc80e7d6b0727ecd421376abc2550283d4"

def reproduce_data_collection(target_path: Path, kaggle_dataset: str) -> None:
    """
    Demonstrates reproducible collection of Attack_Dataset.csv using the Kaggle API,
    with an interactive upload/local mirror fallback and SHA-256 validation.
    """
    target_path.parent.mkdir(parents=True, exist_ok=True)
    
    # 1. Check if already present in target cache
    if target_path.exists():
        print(f"Raw dataset already present at: {target_path}")
    else:
        # 2. Attempt collection via Kaggle API
        kaggle_downloaded = False
        try:
            from kaggle.api.kaggle_api_extended import KaggleApi
            api = KaggleApi()
            api.authenticate()
            print(f"Fetching raw dataset from Kaggle ({kaggle_dataset})...")
            api.dataset_download_files(kaggle_dataset, path=str(target_path.parent), unzip=True)
            if target_path.exists():
                kaggle_downloaded = True
                print(f"Successfully collected via Kaggle API: {target_path}")
            else:
                for p in target_path.parent.glob("*.csv"):
                    if "attack" in p.name.lower():
                        shutil.move(p, target_path)
                        kaggle_downloaded = True
                        break
        except Exception as e:
            print(f"Kaggle API notice: {e}")
            
        # 3. Fallback: Check project repository or prompt for upload
        if not kaggle_downloaded and not target_path.exists():
            print("Using collection fallback: checking project repository...")
            repo_paths = [
                BASE_DIR / "Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv",
                Path("Data Handling and Infrastructure/Milestones/Milestone 1/Attack_Dataset.csv"),
                Path("Attack_Dataset.csv")
            ]
            for rp in repo_paths:
                if rp.exists():
                    print(f"Staging dataset from project repository mirror: {rp}")
                    shutil.copy(rp, target_path)
                    break
                    
        # 4. Fallback: Direct upload if in Colab and still missing
        if not target_path.exists():
            try:
                from google.colab import files
                print("Kaggle API not authenticated. Please upload 'Attack_Dataset.csv' as fallback:")
                uploaded = files.upload()
                for fname, content in uploaded.items():
                    dest = target_path.parent / fname
                    with open(dest, "wb") as f:
                        f.write(content)
                    if fname != target_path.name and not target_path.exists():
                        shutil.copy(dest, target_path)
            except Exception:
                print(f"Could not auto-collect. Please place 'Attack_Dataset.csv' into '{target_path.parent}'.")

    # 5. Validate file integrity against recorded SHA-256 hash (Section 4)
    if target_path.exists():
        hasher = hashlib.sha256()
        with open(target_path, "rb") as f:
            while chunk := f.read(65536):
                hasher.update(chunk)
        calc_hash = hasher.hexdigest()
        print(f"Calculated SHA-256: {calc_hash}")
        if calc_hash == EXPECTED_SHA256:
            print("Integrity check PASSED: Dataset matches original bit-for-bit.")
        else:
            print(f"Notice: Checksum differs from expected {EXPECTED_SHA256[:16]}...")

print(f"Collection target path: {RAW_DATA_FILE}")
print(f"Target Kaggle URL:      {KAGGLE_URL}")
reproduce_data_collection(RAW_DATA_FILE, KAGGLE_DATASET)

## 10. Reproducibility of Preprocessing

## Preprocessing Pipeline Overview

In order for another student or an additional peer reviewer to produce the exact same processed feature data set from raw data, the preprocessing pipeline is made up of 5 sequential and deterministic steps:

## Step 1: Cleaning & Deduplication

- **Removing Duplicates:** The duplicate attack scenarios present in the dataset were deleted using `df.drop_duplicates(subset=["Title", "Scenario Description"])`
- **Formatting Column Names:** Trailing white space was trimmed from each column name header (i.e., ‘Attack Steps ’), and all trailing empty unnamed columns (i.e., “Unnamed: 15”) were also eliminated.
- **Imputing Missing Values:** All missing text values within `Vulnerability`, `Tools Used`, `Target Type`, or `Solution` were replaced with either `"Unknown"`, or empty strings, so that vectorizers and encoders will be able to operate without generating null pointer exceptions.

## Step 2: Feature Engineering & Skewness Handling

- Quantitative signals were created based upon the unstructured text elements: 
• `scenario_len`: Character length of the scenario description.
• `tool_count`: Number of security tools mentioned in the scenario description.
- Since the number of characters in the descriptions, and the number of tools listed may both exhibit right skewness with large numbers of extreme outliers, I used a logarithmic transformation `log1p(x) = log(1 + x)` (Lectures 1 & 2) to provide stable variance across the distribution of the two features.

## Step 3: Cleaning Targets & Categories

- **Cleaning target categories:** Leading/trailing white space was stripped from the `Category` column. Subcategories having less than ten records were combined together into `“Other Specialized Threats”` to ensure that there would be no single sample fold errors during cross validation.
- **Encoding categorical variables:** Nominal metadata features such as `Target Type` and `Source` were encoded into numerical indicators.

## Step 4: Leakage-free Feature Scaling & Text Vectorization

- Text vectorization (TF-IDF on `Scenario Description`) and numerical scaling (`StandardScaler` on derived lengths) are trained **only** on the training subset (`X_train`) to avoid any potential data leakage (Lecture 2 S

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
import numpy as np
import pandas as pd

def run_preprocessing_pipeline(df: pd.DataFrame, is_training: bool = True, vectorizer: TfidfVectorizer = None, scaler: StandardScaler = None):
    """
    End-to-end reproducible preprocessing pipeline for Attack_Dataset.csv.
    Cleans metadata, engineers text lengths, applies TF-IDF, and scales features without data leakage.
    """
    df_clean = df.copy()
    
    # 1. Clean column headers and remove empty trailing columns
    df_clean.columns = [c.strip() for c in df_clean.columns]
    if "Unnamed: 15" in df_clean.columns:
        df_clean = df_clean.drop(columns=["Unnamed: 15"])
        
    # 2. Deduplicate attack records
    df_clean = df_clean.drop_duplicates(subset=["Title", "Scenario Description"])
    
    # 3. Impute missing values in text and categorical fields
    for text_col in ["Scenario Description", "Vulnerability", "Tools Used", "Attack Steps", "Target Type"]:
        if text_col in df_clean.columns:
            df_clean[text_col] = df_clean[text_col].fillna("Unknown")
            
    # 4. Feature engineering: extract length and count metrics
    df_clean["scenario_len"] = df_clean["Scenario Description"].astype(str).str.len()
    df_clean["tool_count"] = df_clean["Tools Used"].apply(
        lambda t: len(str(t).split(",")) if str(t) != "Unknown" else 0
    )
    
    # Log-transform skewed numerical metrics
    for num_col in ["scenario_len", "tool_count"]:
        df_clean[num_col] = np.log1p(np.maximum(df_clean[num_col].values, 0))
        
    # 5. Extract text features via TF-IDF (fit ONLY on training data to prevent leakage)
    text_data = df_clean["Scenario Description"].values
    if is_training:
        vectorizer = TfidfVectorizer(max_features=50, stop_words="english")
        tfidf_features = vectorizer.fit_transform(text_data).toarray()
    else:
        if vectorizer is not None:
            tfidf_features = vectorizer.transform(text_data).toarray()
        else:
            tfidf_features = np.zeros((len(df_clean), 50))
            
    # 6. Fit or apply numerical feature scaler
    num_features = df_clean[["scenario_len", "tool_count"]].values
    if is_training:
        scaler = StandardScaler()
        scaled_num = scaler.fit_transform(num_features)
    else:
        if scaler is not None:
            scaled_num = scaler.transform(num_features)
        else:
            scaled_num = num_features
            
    # Combine numerical features and TF-IDF features
    X_processed = np.hstack([scaled_num, tfidf_features])
    
    return X_processed, vectorizer, scaler

print("Preprocessing pipeline function 'run_preprocessing_pipeline' compiled successfully for Attack_Dataset.csv.")

if RAW_DATA_FILE.exists():
    df_raw = pd.read_csv(RAW_DATA_FILE)
    X_proc, fitted_vec, fitted_scale = run_preprocessing_pipeline(df_raw, is_training=True)
    print(f"Pipeline verification run complete: Feature matrix shape = {X_proc.shape}")